# Step 2A - Tiny Runtime Smoke and G0 Coverage

This notebook is a GPU/runtime preflight for `counterfact_direction1_v1`. It runs only on `dev_tune_200` artifacts and does not touch `analysis_500`, `ablation_500`, or any final-test split.

It checks:

```text
1. Colab can load GSAI-ML/LLaDA-8B-Base with the selected runtime settings.
2. The runtime evaluator can read the frozen protocol JSONL.
3. Candidate-set coverage is written and target_candidate_insert guarantees target support.
4. The sprint-1 method registry executes on a tiny dev-only slice.
```

This is not the full base/self-consistency experiment and not a scientific comparison.

In [ ]:
%%bash
set -euo pipefail

# Colab usually already provides CUDA-compatible torch, so do not reinstall
# unpinned torch unless the environment is missing it.
pip install -q   "transformers==4.46.3"   "datasets>=4.0.0"   "accelerate>=1.11.0"   "sentencepiece>=0.2.0"   "packaging"   "bitsandbytes>=0.43.0"

In [ ]:
import subprocess
import torch, transformers, datasets, accelerate

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)

assert torch.cuda.is_available(), "This notebook needs a GPU runtime."
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
from pathlib import Path
import torch, transformers, datasets, accelerate

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
assert project.exists(), f"Project directory missing after Drive mount: {project}"

env = {
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "accelerate": accelerate.__version__,
}
out = project / "runs/counterfact_direction1_v1/runtime_environment_step2a.json"
out.parent.mkdir(parents=True, exist_ok=True)
with out.open("w", encoding="utf-8") as f:
    json.dump(env, f, indent=2)

print("Wrote:", out)
print(json.dumps(env, indent=2))

## Preflight: Project and Protocol Files

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_DIR = Path("/content/drive/MyDrive/Masters/SB V2/SB")
assert PROJECT_DIR.exists(), f"Missing project dir: {PROJECT_DIR}"

required_files = [
    "llada_runtime_editor_eval.py",
    "llada_counterfact_protocol.py",
    "paper_protocol.md",
    "implementation_spec.md",
    "runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl",
    "runs/counterfact_direction1_v1/protocol/protocol_manifest.json",
]

for name in required_files:
    file_path = PROJECT_DIR / name
    assert file_path.exists(), f"Missing required file: {file_path}"

try:
    commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=PROJECT_DIR,
        text=True,
    ).strip()
    print("git commit:", commit)
except Exception as exc:
    print("git commit unavailable:", repr(exc))

with (PROJECT_DIR / "runs/counterfact_direction1_v1/protocol/protocol_manifest.json").open() as f:
    manifest = json.load(f)

assert manifest["protocol_version"] == "counterfact_direction1_v1"
assert manifest["artifacts"]["dev_tune_200"]["summary"]["count"] == 200

print("Python:", sys.version)
print("Project dir OK:", PROJECT_DIR)
print("Protocol OK:", manifest["protocol_version"])

## Create A Better Two-Edit Smoke Slice

Use one one-token edit and one multi-token edit so the smoke run exercises short-span handling.

In [ ]:
from pathlib import Path
import json

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
src = project / "runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl"
smoke_dir = project / "runs/counterfact_direction1_v1/smoke_inputs"
smoke_dir.mkdir(parents=True, exist_ok=True)
dst = smoke_dir / "dev_tune_smoke_2.jsonl"

rows = [json.loads(line) for line in src.open()]
one_token = next(r for r in rows if str(r.get("target_length_bin")) == "1")
multi_token = next(
    r for r in rows
    if str(r.get("target_length_bin")) in {"2", "3", "4", ">=4", "4+"}
)

with dst.open("w", encoding="utf-8") as f:
    print(json.dumps(one_token, ensure_ascii=False), file=f)
    print(json.dumps(multi_token, ensure_ascii=False), file=f)

print("Wrote derived smoke input:", dst)
print("This is intentionally outside the frozen protocol/ directory.")
print("case_ids:", one_token["case_id"], multi_token["case_id"])
print("bins:", one_token["target_length_bin"], multi_token["target_length_bin"])

## Overwrite Guards

These runs become part of the experiment trail. Delete a run directory manually or choose a new name if you need to rerun.

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
run_dirs = [
    project / "runs/counterfact_direction1_v1/dev_tune_200_smoke_base",
    project / "runs/counterfact_direction1_v1/dev_tune_200_smoke_methods",
    project / "runs/counterfact_direction1_v1/dev_tune_200_smoke_mc_bridge",
    project / "runs/counterfact_direction1_v1/dev_tune_200_smoke_raw_bridge_gated",
    project / "runs/counterfact_direction1_v1/dev_tune_200_base_coverage",
]

for d in run_dirs:
    if d.exists():
        raise RuntimeError(
            f"Run directory already exists: {d}. "
            "Delete it manually or create a new run name. Do not overwrite silently."
        )

print("Overwrite guard passed.")

## Step 2A.1: Tiny Base Runtime Smoke

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_smoke_base

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/smoke_inputs/dev_tune_smoke_2.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_smoke_base \
  --methods base \
  --max_edits 2 \
  --eval_samples 1 \
  --steps 2 \
  --bridge_topk 4 \
  --mc_rollouts 1 \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_smoke_base/stdout.log

### Inspect Tiny Base Coverage

In [ ]:
import json
from pathlib import Path

path = Path("/content/drive/MyDrive/Masters/SB V2/SB/runs/counterfact_direction1_v1/dev_tune_200_smoke_base/candidate_coverage.jsonl")
assert path.exists(), f"Missing candidate coverage file: {path}"
rows = [json.loads(line) for line in path.open()]
assert len(rows) == 2, f"Expected 2 smoke rows, got {len(rows)}"

required_keys = [
    "target_new_first_token_in_base_topk",
    "all_target_new_tokens_in_base_topk",
    "target_true_first_token_in_base_topk",
    "all_target_true_tokens_in_base_topk",
    "all_target_new_tokens_after_candidate_insert",
]
for row in rows:
    for key in required_keys:
        assert key in row, f"Missing key {key} in row: {row.keys()}"

assert all(bool(r["all_target_new_tokens_after_candidate_insert"]) for r in rows), \
    "Candidate insertion did not guarantee target_new support in smoke run."

print("num edits:", len(rows))
for key in required_keys:
    print(key, sum(bool(r[key]) for r in rows), "/", len(rows))
print("Tiny smoke candidate coverage passed.")

## Step 2A.2: Tiny Method-Registry Smoke

This is an implementation smoke only. It checks that first-sprint non-MC methods execute without crashing.

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_smoke_methods

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/smoke_inputs/dev_tune_smoke_2.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_smoke_methods \
  --methods target_logit_bias,target_candidate_insert,prompt_memory,myopic_score,no_rollout_bridge \
  --max_edits 2 \
  --eval_samples 1 \
  --steps 2 \
  --bridge_topk 4 \
  --mc_rollouts 0 \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_smoke_methods/stdout.log

## Step 2A.3: Tiny MC Bridge Smoke

This verifies the MC rollout path. It is not a performance result.

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_smoke_mc_bridge

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/smoke_inputs/dev_tune_smoke_2.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_smoke_mc_bridge \
  --methods mc_bridge \
  --max_edits 2 \
  --eval_samples 1 \
  --steps 2 \
  --bridge_topk 4 \
  --mc_rollouts 1 \
  --guidance_scale 1.0 \
  --reward_mode soft_overlap \
  --reward_beta 6.0 \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_smoke_mc_bridge/stdout.log

## Step 2A.4: Tiny Raw Bridge Gated Smoke

This verifies the gated runtime bridge path with a harmless subject gate. It is still an implementation smoke, not a scientific result.

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_smoke_raw_bridge_gated

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/smoke_inputs/dev_tune_smoke_2.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_smoke_raw_bridge_gated \
  --methods raw_bridge_gated \
  --max_edits 2 \
  --eval_samples 1 \
  --steps 2 \
  --bridge_topk 4 \
  --mc_rollouts 1 \
  --guidance_scale 1.0 \
  --reward_mode soft_overlap \
  --reward_beta 6.0 \
  --gate_mode subject \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_smoke_raw_bridge_gated/stdout.log

## Assert Runtime Run Configs

Every runtime output must preserve the frozen protocol access assumptions.

In [ ]:
import json
from pathlib import Path

def load_run_config(run_dir):
    run_dir = Path(run_dir)
    candidates = [
        run_dir / "run_config.json",
        run_dir / "config_snapshot.json",
        run_dir / "config.json",
    ]
    for path in candidates:
        if path.exists():
            with path.open() as f:
                return json.load(f), path
    raise AssertionError(f"No run config found in {run_dir}")

def assert_protocol_config(run_dir):
    cfg, path = load_run_config(run_dir)
    expected = {
        "protocol_version": "counterfact_direction1_v1",
        "edit_access": "given_at_edit_time",
        "training_access": "none",
        "hyperparameter_access": "dev_tune_only",
    }
    for key, value in expected.items():
        assert cfg.get(key) == value, (
            f"{path}: expected {key}={value!r}, got {cfg.get(key)!r}"
        )
    print("Protocol config OK:", path)

base = Path("/content/drive/MyDrive/Masters/SB V2/SB/runs/counterfact_direction1_v1")
assert_protocol_config(base / "dev_tune_200_smoke_base")
assert_protocol_config(base / "dev_tune_200_smoke_methods")
assert_protocol_config(base / "dev_tune_200_smoke_mc_bridge")
assert_protocol_config(base / "dev_tune_200_smoke_raw_bridge_gated")

## Step 2B: Coverage-Only G0 On Full `dev_tune_200`

This writes G0 candidate support for all 200 dev edits without prompt generation. It is cheaper than the base evaluation pass and is the right artifact to inspect before bridge comparisons.

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_base_coverage

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_base_coverage \
  --methods base \
  --coverage_only 1 \
  --eval_samples 1 \
  --steps 4 \
  --bridge_topk 4 \
  --mc_rollouts 1 \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_base_coverage/stdout.log

### Inspect Full G0 Coverage

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

path = Path("/content/drive/MyDrive/Masters/SB V2/SB/runs/counterfact_direction1_v1/dev_tune_200_base_coverage/candidate_coverage.jsonl")
assert path.exists(), f"Missing candidate coverage file: {path}"
rows = [json.loads(line) for line in path.open()]
assert len(rows) == 200, f"Expected 200 dev_tune rows, got {len(rows)}"
assert all(bool(r["all_target_new_tokens_after_candidate_insert"]) for r in rows), \
    "Candidate insertion failed for at least one dev_tune_200 edit."

def bin_key(row):
    return str(row["target_length_bin"])

bin_order = {"1": 0, "2": 1, "3": 2, ">=4": 3, "4+": 3}
by_len = defaultdict(list)
for row in rows:
    by_len[bin_key(row)].append(row)

ordered_bins = sorted(by_len, key=lambda x: bin_order.get(str(x), 99))
print("num edits:", len(rows))
print("target length bins:", {k: len(by_len[k]) for k in ordered_bins})

for key in [
    "target_new_first_token_in_base_topk",
    "all_target_new_tokens_in_base_topk",
    "target_true_first_token_in_base_topk",
    "all_target_true_tokens_in_base_topk",
    "all_target_new_tokens_after_candidate_insert",
]:
    c = sum(bool(r[key]) for r in rows)
    print(key, c, "/", len(rows), f"({c/len(rows):.3f})")

print("\nBy target length:")
for length_bin in ordered_bins:
    group = by_len[length_bin]
    c = sum(bool(r["all_target_new_tokens_in_base_topk"]) for r in group)
    print(
        "len", length_bin,
        "n=", len(group),
        "all target_new in base top-k:",
        c, "/", len(group),
    )

assert_protocol_config(Path("/content/drive/MyDrive/Masters/SB V2/SB/runs/counterfact_direction1_v1/dev_tune_200_base_coverage"))
print("Full G0 candidate coverage passed.")

## Expected Outcome

Stop if any assertion fails, if the model cannot load, or if candidate insertion does not guarantee target support.

If all cells pass, Step 2A has cleared the fragile engineering path: GPU runtime, frozen manifests, candidate coverage, method registry, and MC bridge execution. The next notebook should run the actual base/self-consistency experiment on `dev_tune_200`; this notebook is not that final Experiment 1.